In [1]:
%matplotlib qt5
%load_ext autoreload
%autoreload 2
    
import SF_RugPeek as sf
    
import numpy as np
import matplotlib.pyplot as plt
from os import listdir

import dynesty

from matplotlib.gridspec import GridSpec
from matplotlib.widgets import Slider, Button

%matplotlib qt5

In [3]:
Sample = 'Mb'
directory = r"C:\Users\tedc4\Documents\Stopped_Flow_UV-VIS\25_\Processed"

listdir(directory+f'\{Sample}')

['40.0uM_Mb_+10_equ-H2O2_processed.dat',
 '40.0uM_Mb_+10_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+20_equ-H2O2_processed.dat',
 '40.0uM_Mb_+20_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+30_equ-H2O2_processed.dat',
 '40.0uM_Mb_+30_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+40_equ-H2O2_processed.dat',
 '40.0uM_Mb_+40_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+50_equ-H2O2_processed.dat',
 '40.0uM_Mb_+50_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+60_equ-H2O2_processed.dat',
 '40.0uM_Mb_+60_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+70_equ-H2O2_processed.dat',
 '40.0uM_Mb_+70_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+80_equ-H2O2_processed.dat',
 '40.0uM_Mb_+80_equ-H2O2_st_dev_arr.dat',
 '40.0uM_Mb_+90_equ-H2O2_processed.dat',
 '40.0uM_Mb_+90_equ-H2O2_st_dev_arr.dat']

In [5]:
sample_dict = {}
sig_dict = {}

sample_fnames = []

for i in range(len(listdir(directory+f'\{Sample}')) // 2):
    
    fname = listdir(directory+f'\{Sample}')[i*2][:listdir(directory+f'\{Sample}')[i*2].rindex('equ')+8]    
    sample_fnames.append(fname)
    
    sample_dict[fname] = sf.SF_Rug(directory=directory+f'\{Sample}', filename=str(listdir(directory+f'\{Sample}')[i*2]))
    sig_dict[f'{fname}_st_dev_arr'] = sf.SF_Rug(directory=directory+f'\{Sample}', filename=str(listdir(directory+f'\{Sample}')[(i*2)+1]))
    

In [7]:
list(sample_dict)

['40.0uM_Mb_+10_equ-H2O2',
 '40.0uM_Mb_+20_equ-H2O2',
 '40.0uM_Mb_+30_equ-H2O2',
 '40.0uM_Mb_+40_equ-H2O2',
 '40.0uM_Mb_+50_equ-H2O2',
 '40.0uM_Mb_+60_equ-H2O2',
 '40.0uM_Mb_+70_equ-H2O2',
 '40.0uM_Mb_+80_equ-H2O2',
 '40.0uM_Mb_+90_equ-H2O2']

In [47]:
## select a set of averaged scans loaded in as an SF_Rug object ##

file_idx = 2

sample = sample_dict[list(sample_dict)[file_idx]]
sample.sig_x =  sig_dict[list(sig_dict)[file_idx]].abs

sample_name = sample.fig_title.replace('_', ' ')

print(f'Selected file: {sample_name}')

Selected file: 40.0uM Mb +30 equ-H2O2


In [43]:
sample.explore_spectra()

In [105]:
sample.explore_traces(ax_min=-10)

In [41]:
## compute the SVD and generagte a plot of the eigenvalues and left and right eigenvectors ##
plt.close('all')

#sample.compute_SVD(threshold=2000)
#sample.explore_SVD(df_idx=file_idx)

# need to make explore SVD accept rug objects generated from raw csv data and averaged data due to change in length of self.metdata
# self.metadata[df_idx] for average arr
# self.metadata[df_idx][-1] for csv data

In [49]:
## construct the transfer matrix E ##

E = np.array([[0, 1],
              [0, 1]])

## specify the idx (in evals) of any rate coefficients for decay pathways that populate stable, absorbing products ##
Ep = None # [-1] 

## input the ratio of the initial compartment populations ##
C0 = np.array([1, 0]).T

sigma = 1 # sample.sig_x

# 10 equ. 
lowlim = np.array([1e-2, 1e2])
uplim = np.array([1e2, 1e4])

taulims = (lowlim, uplim)

## multiple c in case one compartment degrades to a spectroscopically inactive compartment i.e. from peroxide degradation
c_ll = np.array([])
c_ul = np.array([])

c_lims = (c_ll, c_ul)



In [33]:
sample.__dict__.keys()

dict_keys(['wavelengths', 'delays', 'abs', 'fig_title', 'metadata', 'singular_values', 'principal_spectra', 'principal_kinetics', 'singular_fractions', 'relevant_sing', 'relevant_spectra', 'relevant_kinetics', 'relevant_matrices', 'reconstructed'])

In [51]:
%autoreload 2
## run the dynamic nested Sampler ##

save_directory = r"C:\Users\tedc4\Documents\Stopped_Flow_UV-VIS\25_\DNS_DATA"
# modeltypes = ['\Competitive', '\Sequential', '\Parallel']

modeltype = '\Sequential'

# sample = sample_ # 4
filename = sample_name.replace(' ', '_')
filename = filename.replace('.0', '')

suffix = 0

## p_frac == 0: 100% weight on evidence, 0% on posterior ##
p_frac = 0.5

vary_c=False


plt.close('all')

if hasattr(sample, 'HS_conc'):
    HS_conc = sample.HS_conc[df_idx]
else:
    HS_conc = 0


print(f'save directory = {save_directory}{modeltype}\{Sample}\nfilename = {E.shape[0]}-cpmt_{np.count_nonzero(E)}-k_Ep={Ep}_{modeltype[1:]}_model_{filename}_+{HS_conc}_equ-HS_{sample.delays.shape[0]}-{sample.metadata[-1]}-{int(sample.delays[-1])}s_{c_lims[0].shape[0]}c-vary={vary_c}_DNS_{suffix}.save\n')


sf.SF_Rug.run_nested_sampling({'pfrac': p_frac},
                              {'pfrac': p_frac},
                              E, C0, sample, sigma, taulims, c_lims, Ep=Ep, var_c=vary_c,
                              checkpoint_file=f'{save_directory}{modeltype}\{Sample}\{E.shape[0]}-cpmt_{np.count_nonzero(E)}-k_Ep={Ep}_{modeltype[1:]}_model_{filename}_+{HS_conc}_equ-HS_{sample.delays.shape[0]}-{sample.metadata[-1]}-{int(sample.delays[-1])}s_{c_lims[0].shape[0]}c-vary={vary_c}_DNS_{suffix}.save')
                              # sample_method='auto',
                              # bound_method='multi')


save directory = C:\Users\tedc4\Documents\Stopped_Flow_UV-VIS\25_\DNS_DATA\Sequential\Mb
filename = 2-cpmt_2-k_Ep=None_Sequential_model_40uM_Mb_+30_equ-H2O2_+0_equ-HS_1000-logarithmic-250s_0c-vary=False_DNS_0.save



15455it [15:03, 17.11it/s, batch: 8 | bound: 0 | nc: 1 | ncall: 84094 | eff(%): 18.378 | loglstar:   -inf < -0.752 < -1.370 | logz: -3.636 +/-  0.038 | stop:  0.944]           


Summary
niter: 15455
ncall: 73365
eff(%): 18.378
logz: -3.597 +/-  0.036


In [26]:
plt.figure()
plt.plot(sample.delays, sample.abs)

In [28]:
plt.figure()
plt.plot(sample.delays, sample.sig_x)